# 12 — Final Visuals

This notebook comes after `11_Sensitivity_Analysis.ipynb`.

Purpose: create the final report/presentation visuals and report-ready tables from the already-generated outputs.

This notebook **does not rerun the methodology**. It only reads outputs from notebooks `06–11` and creates clean visuals.

Configurable main choice:

```text
FINAL_PROFILE_LEVEL = 4
```

This keeps 2007 and 2019 comparable and avoids the over-granularity of the 5-level working run. You can switch it to `3` or `5` later if needed.

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import re, shutil, warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
PROFILE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
EPSILON_MARGIN_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Margin_POSet_Robustness"
RECOVERY_VALIDATION_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"
SENSITIVITY_DIR = PROJECT_ROOT / "Data" / "Processed" / "Sensitivity_Analysis"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Final_Visuals"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "12_Final_Visuals"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Final_Visuals"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Final_Visuals"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True
if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, TABLES_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Figures:", FIGURES_DIR.resolve())
print("Tables:", TABLES_DIR.resolve())

Run ID: 20260622_142318
Figures: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Final_Visuals
Tables: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Tables\Final_Visuals


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

MAIN_SET_NAME = "baseline_6_variables"
FINAL_PROFILE_LEVEL = 4
FINAL_EPSILON_MARGIN_FOR_VISUALS = 0.05

SHOCK_IDS = ["2007", "2019"]

PRIMARY_RECOVERY_VARIANT = "all_recovery_available"
STRICT_RECOVERY_VARIANT = "affected_recovered_only_exclude_0_and_20"

SHOW_COUNTRY_CODES_IN_HASSE = True
MAX_COUNTRY_CODES_PER_PROFILE_LABEL = 4
MAX_READABLE_HASSE_NODES = 45
FIG_DPI = 220

print("Main set:", MAIN_SET_NAME)
print("Final profile level:", FINAL_PROFILE_LEVEL)
print("Reference epsilon margin:", FINAL_EPSILON_MARGIN_FOR_VISUALS)

Main set: baseline_6_variables
Final profile level: 4
Reference epsilon margin: 0.05


In [3]:
# ------------------------------------------------------
# HELPERS
# ------------------------------------------------------

def read_csv_if_exists(path, label, required=False):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {label}: {len(df)} rows")
        return df
    if required:
        raise FileNotFoundError(f"{label} not found: {path}")
    print(f"Optional missing: {label} -> {path}")
    return pd.DataFrame()

def clean_keys(df):
    out = df.copy()
    for col in ["Country", "source_country", "target_country"]:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().str.upper()
    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.strip()
    if "profile_levels" in out.columns:
        out["profile_levels"] = pd.to_numeric(out["profile_levels"], errors="coerce")
    if "epsilon_margin" in out.columns:
        out["epsilon_margin"] = pd.to_numeric(out["epsilon_margin"], errors="coerce")
    if "epsilon_value" in out.columns:
        out["epsilon_value"] = pd.to_numeric(out["epsilon_value"], errors="coerce")
    if "Recovery" in out.columns:
        out["Recovery"] = pd.to_numeric(out["Recovery"], errors="coerce")
    return out

figure_inventory = []
table_inventory = []

def save_figure(fig, filename, title, description, source_tables):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    figure_inventory.append({
        "figure_file": filename,
        "figure_path": str(path),
        "title": title,
        "description": description,
        "source_tables": source_tables,
        "created_at": RUN_TIMESTAMP,
    })
    print("Saved figure:", filename)
    return path

def save_table(df, filename, title, description):
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    table_inventory.append({
        "table_file": filename,
        "table_path": str(path),
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "created_at": RUN_TIMESTAMP,
    })
    print("Saved table:", filename)
    return path

def country_label(countries_string):
    countries = [c for c in str(countries_string).split(";") if c and c != "nan"]
    if not SHOW_COUNTRY_CODES_IN_HASSE:
        return ""
    if len(countries) <= MAX_COUNTRY_CODES_PER_PROFILE_LABEL:
        return ", ".join(countries)
    return ", ".join(countries[:MAX_COUNTRY_CODES_PER_PROFILE_LABEL]) + f" +{len(countries)-MAX_COUNTRY_CODES_PER_PROFILE_LABEL}"

def draw_pipeline_diagram(steps, title):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.axis("off")
    xs = np.linspace(0.05, 0.95, len(steps))
    y = 0.5
    for i, (x, step) in enumerate(zip(xs, steps)):
        box = FancyBboxPatch((x-0.06, y-0.12), 0.12, 0.24,
                             boxstyle="round,pad=0.02,rounding_size=0.025",
                             linewidth=1, edgecolor="black", facecolor="white")
        ax.add_patch(box)
        ax.text(x, y, step, ha="center", va="center", fontsize=9, wrap=True)
        if i < len(steps)-1:
            ax.annotate("", xy=(xs[i+1]-0.075, y), xytext=(x+0.075, y),
                        arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.set_title(title, fontsize=14, pad=18)
    return fig

def build_hasse_positions(profile_table):
    positions = {}
    pt = profile_table.copy()
    pt["poset_layer"] = pd.to_numeric(pt["poset_layer"], errors="coerce")
    for layer, group in pt.groupby("poset_layer"):
        group = group.sort_values(["profile_code", "profile_id"]).reset_index(drop=True)
        count = len(group)
        xs = [0.0] if count == 1 else np.linspace(-count/2, count/2, count)
        y = -float(layer)
        for x, (_, row) in zip(xs, group.iterrows()):
            positions[row["profile_id"]] = (float(x), y)
    return positions

def draw_hasse_diagram(profile_table, hasse_edges, title):
    pt = profile_table.copy()
    edges = hasse_edges.copy()
    pt["poset_layer"] = pd.to_numeric(pt["poset_layer"], errors="coerce")
    max_layer = int(pt["poset_layer"].max()) if not pt.empty else 1
    fig_height = max(7, max_layer * 1.6)
    fig_width = max(12, min(20, len(pt) * 0.75))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    positions = build_hasse_positions(pt)

    for _, edge in edges.iterrows():
        s = edge.get("source_profile_id")
        t = edge.get("target_profile_id")
        if s in positions and t in positions:
            x1, y1 = positions[s]
            x2, y2 = positions[t]
            ax.annotate("", xy=(x2, y2+0.08), xytext=(x1, y1-0.08),
                        arrowprops=dict(arrowstyle="->", lw=0.8, alpha=0.65))

    for _, row in pt.iterrows():
        pid = row["profile_id"]
        x, y = positions[pid]
        countries = country_label(row.get("countries", ""))
        parts = [str(pid), str(row.get("profile_code", "")), f"n={int(row.get('country_count', 0))}"]
        if countries:
            parts.append(countries)
        label = "\n".join(parts)
        ax.scatter(x, y, s=950, zorder=3)
        ax.text(x, y, label, ha="center", va="center", fontsize=7.5, zorder=4)

    ax.set_title(title, fontsize=14, pad=18)
    ax.set_xlabel("Profiles within layer")
    ax.set_ylabel("POSet layer; 1 = nondominated / Pareto")
    ax.grid(axis="y", alpha=0.2)
    yticks = sorted([-float(layer) for layer in pt["poset_layer"].dropna().unique()])
    ax.set_yticks(yticks)
    ax.set_yticklabels([str(int(abs(y))) for y in yticks])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return fig

In [4]:
# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

candidate_variable_scorecard = read_csv_if_exists(PRE_POSET_DIR / "candidate_variable_scorecard.csv", "candidate_variable_scorecard")
candidate_variable_redundancy_pairs = read_csv_if_exists(PRE_POSET_DIR / "candidate_variable_redundancy_pairs.csv", "candidate_variable_redundancy_pairs")

profile_table = read_csv_if_exists(PROFILE_POSET_DIR / "profile_table.csv", "profile_table", required=True)
profile_country_map = read_csv_if_exists(PROFILE_POSET_DIR / "profile_country_map.csv", "profile_country_map", required=True)
profile_hasse_edges = read_csv_if_exists(PROFILE_POSET_DIR / "profile_hasse_edges.csv", "profile_hasse_edges", required=True)
profile_run_summary = read_csv_if_exists(PROFILE_POSET_DIR / "profile_run_summary.csv", "profile_run_summary", required=True)

epsilon_margin_run_summary = read_csv_if_exists(EPSILON_MARGIN_DIR / "epsilon_margin_run_summary.csv", "epsilon_margin_run_summary")
epsilon_rule_comparison = read_csv_if_exists(EPSILON_MARGIN_DIR / "epsilon_tolerance_vs_margin_comparison.csv", "epsilon_tolerance_vs_margin_comparison")

recovery_takeaway = read_csv_if_exists(RECOVERY_VALIDATION_DIR / "recovery_validation_takeaway_table.csv", "recovery_validation_takeaway_table")
recovery_interpretation_cases = read_csv_if_exists(RECOVERY_VALIDATION_DIR / "recovery_interpretation_cases.csv", "recovery_interpretation_cases")

sensitivity_synthesis = read_csv_if_exists(SENSITIVITY_DIR / "sensitivity_synthesis.csv", "sensitivity_synthesis")
methodological_decisions = read_csv_if_exists(SENSITIVITY_DIR / "methodological_decision_candidates.csv", "methodological_decision_candidates")
report_table_profile_sensitivity = read_csv_if_exists(SENSITIVITY_DIR / "report_table_profile_sensitivity.csv", "report_table_profile_sensitivity")
report_table_epsilon_margin = read_csv_if_exists(SENSITIVITY_DIR / "report_table_epsilon_margin.csv", "report_table_epsilon_margin")
report_table_recovery_validation = read_csv_if_exists(SENSITIVITY_DIR / "report_table_recovery_validation.csv", "report_table_recovery_validation")

for name in [
    "candidate_variable_scorecard", "candidate_variable_redundancy_pairs",
    "profile_table", "profile_country_map", "profile_hasse_edges", "profile_run_summary",
    "epsilon_margin_run_summary", "epsilon_rule_comparison",
    "recovery_takeaway", "recovery_interpretation_cases",
    "sensitivity_synthesis", "methodological_decisions",
    "report_table_profile_sensitivity", "report_table_epsilon_margin", "report_table_recovery_validation",
]:
    df = globals()[name]
    if isinstance(df, pd.DataFrame) and not df.empty:
        globals()[name] = clean_keys(df)

input_summary = pd.DataFrame([
    {"input": "profile_table", "rows": len(profile_table), "columns": len(profile_table.columns)},
    {"input": "profile_country_map", "rows": len(profile_country_map), "columns": len(profile_country_map.columns)},
    {"input": "profile_hasse_edges", "rows": len(profile_hasse_edges), "columns": len(profile_hasse_edges.columns)},
    {"input": "profile_run_summary", "rows": len(profile_run_summary), "columns": len(profile_run_summary.columns)},
    {"input": "epsilon_margin_run_summary", "rows": len(epsilon_margin_run_summary), "columns": len(epsilon_margin_run_summary.columns)},
    {"input": "report_table_recovery_validation", "rows": len(report_table_recovery_validation), "columns": len(report_table_recovery_validation.columns)},
    {"input": "sensitivity_synthesis", "rows": len(sensitivity_synthesis), "columns": len(sensitivity_synthesis.columns)},
])
input_summary.to_csv(DIAGNOSTICS_DIR / "final_visuals_input_summary.csv", index=False)
display(input_summary)

Loaded candidate_variable_scorecard: 16 rows
Optional missing: candidate_variable_redundancy_pairs -> D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Pre_POSet_EDA\candidate_variable_redundancy_pairs.csv
Loaded profile_table: 341 rows
Loaded profile_country_map: 349 rows
Loaded profile_hasse_edges: 712 rows
Loaded profile_run_summary: 12 rows
Loaded epsilon_margin_run_summary: 48 rows
Loaded epsilon_tolerance_vs_margin_comparison: 48 rows
Loaded recovery_validation_takeaway_table: 16 rows
Loaded recovery_interpretation_cases: 22 rows
Loaded sensitivity_synthesis: 5 rows
Loaded methodological_decision_candidates: 5 rows
Loaded report_table_profile_sensitivity: 6 rows
Loaded report_table_epsilon_margin: 12 rows
Loaded report_table_recovery_validation: 16 rows


,input,rows,columns
0,profile_table,341,39
1,profile_country_map,349,46
2,profile_hasse_edges,712,12
3,profile_run_summary,12,23
4,epsilon_margin_run_summary,48,23
5,report_table_recovery_validation,16,14
6,sensitivity_synthesis,5,4


In [5]:
# ------------------------------------------------------
# SELECT FINAL PROFILE RUN
# ------------------------------------------------------

final_profile_table = profile_table[
    (profile_table["set_name"] == MAIN_SET_NAME)
    & (profile_table["profile_levels"] == FINAL_PROFILE_LEVEL)
].copy()

final_profile_country_map = profile_country_map[
    (profile_country_map["set_name"] == MAIN_SET_NAME)
    & (profile_country_map["profile_levels"] == FINAL_PROFILE_LEVEL)
].copy()

final_profile_hasse_edges = profile_hasse_edges[
    (profile_hasse_edges["set_name"] == MAIN_SET_NAME)
    & (profile_hasse_edges["profile_levels"] == FINAL_PROFILE_LEVEL)
].copy()

final_profile_run_summary = profile_run_summary[
    (profile_run_summary["set_name"] == MAIN_SET_NAME)
    & (profile_run_summary["profile_levels"] == FINAL_PROFILE_LEVEL)
].copy()

if final_profile_table.empty:
    raise ValueError(
        f"No profile table rows found for {MAIN_SET_NAME} at {FINAL_PROFILE_LEVEL} levels. "
        "Change FINAL_PROFILE_LEVEL or rerun notebook 07."
    )

final_profile_summary = pd.DataFrame([{
    "main_set_name": MAIN_SET_NAME,
    "final_profile_level": FINAL_PROFILE_LEVEL,
    "profile_rows": len(final_profile_table),
    "country_rows": len(final_profile_country_map),
    "hasse_edges": len(final_profile_hasse_edges),
    "shock_ids": ",".join(sorted(final_profile_table["shock_id"].astype(str).unique())),
    "readability_warning": "review" if final_profile_table.groupby("shock_id").size().max() > MAX_READABLE_HASSE_NODES else "ok",
}])
final_profile_summary.to_csv(PROCESSED_DIR / "final_profile_visual_selection_summary.csv", index=False)
display(final_profile_summary)
display(final_profile_run_summary)

,main_set_name,final_profile_level,profile_rows,country_rows,hasse_edges,shock_ids,readability_warning
0,baseline_6_variables,4,56,57,113,"2007,2019",ok


,run_key,set_name,shock_id,profile_levels,sample_strategy,require_recovery_available,country_count,profile_count,variable_count,variables,pareto_profile_count,pareto_country_count,dominance_relation_count,hasse_edge_count,layer_count,max_country_count_per_profile,mean_country_count_per_profile,comparable_pairs,incomparable_pairs,comparability_ratio,incomparability_ratio,is_working_main_run,hasse_diagram_path
1,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,complete_case,True,25,25,6,"debt_capacity_score_0_100,employment_strength_...",8,8,83,51,5,1,1.0000,83,217,0.2767,0.7233,False,D:\Milano Bicocca\Course Materials\1st Year\02...
4,baseline_6_variables__shock_2019__levels_4,baseline_6_variables,2019,4,complete_case,True,32,31,6,"debt_capacity_score_0_100,employment_strength_...",13,13,126,62,5,2,1.0323,126,339,0.2710,0.7290,False,D:\Milano Bicocca\Course Materials\1st Year\02...


In [6]:
# ------------------------------------------------------
# FIGURE 01 — PROJECT PIPELINE
# ------------------------------------------------------

steps = [
    "Raw macro, energy, governance data",
    "Country-year standardization",
    "Recovery outcome construction",
    "Direction-aligned structural variables",
    "Profile POSet / Pareto analysis",
    "Epsilon robustness",
    "Recovery validation",
    "Sensitivity + final visuals",
]
fig = draw_pipeline_diagram(steps, "Project pipeline: economic resilience as multidimensional structural capacity")
save_figure(fig, "01_project_pipeline_diagram.png", "Project pipeline", "High-level project pipeline.", "notebooks 00–12")

Saved figure: 01_project_pipeline_diagram.png


WindowsPath('D:/Milano Bicocca/Course Materials/1st Year/02 - Data Science Lab - 2526-1-FDS02Q003/03 Projects/Index/Notebooks/Outputs/Figures/Final_Visuals/01_project_pipeline_diagram.png')

In [7]:
# ------------------------------------------------------
# TABLE 01 + FIGURE 02 — VARIABLE CONCEPT MAP
# ------------------------------------------------------

final_variable_concepts = pd.DataFrame([
    {"ordering_variable":"debt_capacity_score_0_100","concept":"Fiscal capacity / debt room","source_construct":"Public debt / GDP, inverted","direction":"Higher = better","role":"baseline structural variable"},
    {"ordering_variable":"employment_strength_score_0_100","concept":"Labour-market strength","source_construct":"Unemployment rate, inverted","direction":"Higher = better","role":"baseline structural variable"},
    {"ordering_variable":"rd_intensity_score_0_100","concept":"Innovation capacity","source_construct":"R&D expenditure / GDP","direction":"Higher = better","role":"baseline structural variable"},
    {"ordering_variable":"human_capital_tertiary_score_0_100","concept":"Human-capital stock","source_construct":"Adult tertiary educational attainment, age 25–64","direction":"Higher = better","role":"baseline structural variable"},
    {"ordering_variable":"energy_security_score_0_100","concept":"Energy security","source_construct":"Energy import dependency, inverted","direction":"Higher = better","role":"baseline structural variable"},
    {"ordering_variable":"governance_capacity_score_0_100","concept":"Governance capacity","source_construct":"Derived Mahalanobis WGI composite","direction":"Higher = better","role":"baseline structural variable"},
])
save_table(final_variable_concepts, "01_final_variable_concepts.csv", "Final variable concepts", "Conceptual mapping of the six baseline variables.")

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")
concepts = final_variable_concepts["concept"].tolist()
angles = np.linspace(0, 2*np.pi, len(concepts), endpoint=False)
ax.scatter([0], [0], s=1800)
ax.text(0, 0, "Structural\nresilience\ncapacity", ha="center", va="center", fontsize=10)
for angle, concept in zip(angles, concepts):
    x, y = np.cos(angle), np.sin(angle)
    ax.plot([0, x*0.78], [0, y*0.78], lw=1)
    ax.scatter([x], [y], s=950)
    ax.text(x, y, concept, ha="center", va="center", fontsize=8, wrap=True)
ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.2, 1.2)
ax.set_title("Six-dimensional structural resilience framework", fontsize=14, pad=18)
save_figure(fig, "02_structural_variable_concept_map.png", "Structural variable concept map", "Six baseline dimensions.", "01_final_variable_concepts.csv")

Saved table: 01_final_variable_concepts.csv
Saved figure: 02_structural_variable_concept_map.png


WindowsPath('D:/Milano Bicocca/Course Materials/1st Year/02 - Data Science Lab - 2526-1-FDS02Q003/03 Projects/Index/Notebooks/Outputs/Figures/Final_Visuals/02_structural_variable_concept_map.png')

In [8]:
# ------------------------------------------------------
# FIGURES 03A/03B — FINAL Hasse diagrams
# ------------------------------------------------------

for shock_id in SHOCK_IDS:
    pt = final_profile_table[final_profile_table["shock_id"].astype(str) == str(shock_id)].copy()
    edges = final_profile_hasse_edges[final_profile_hasse_edges["shock_id"].astype(str) == str(shock_id)].copy()

    if pt.empty:
        print(f"No profiles for shock {shock_id}; skipping.")
        continue

    fig = draw_hasse_diagram(pt, edges, f"Profile POSet Hasse diagram — shock {shock_id}, {FINAL_PROFILE_LEVEL} levels")
    save_figure(
        fig,
        f"03_profile_hasse_diagram_shock_{shock_id}_{FINAL_PROFILE_LEVEL}levels.png",
        f"Profile Hasse diagram — {shock_id}",
        f"Transitive-reduced Hasse diagram for {FINAL_PROFILE_LEVEL}-level baseline profile POSet.",
        "profile_table.csv; profile_hasse_edges.csv",
    )

Saved figure: 03_profile_hasse_diagram_shock_2007_4levels.png
Saved figure: 03_profile_hasse_diagram_shock_2019_4levels.png


In [9]:
# ------------------------------------------------------
# FIGURE 04 — PROFILE STRUCTURE SUMMARY
# ------------------------------------------------------

summary_cols = ["shock_id","country_count","profile_count","pareto_country_count","comparability_ratio","incomparability_ratio","layer_count"]
profile_summary_plot_df = final_profile_run_summary[summary_cols].copy()
profile_summary_plot_df["pareto_share"] = profile_summary_plot_df["pareto_country_count"] / profile_summary_plot_df["country_count"]

save_table(profile_summary_plot_df, "02_final_profile_summary_by_shock.csv", "Final profile summary by shock", "Selected final profile-level metrics.")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(profile_summary_plot_df))
width = 0.35
ax.bar(x - width/2, profile_summary_plot_df["comparability_ratio"], width, label="Comparability")
ax.bar(x + width/2, profile_summary_plot_df["pareto_share"], width, label="Pareto share")
ax.set_xticks(x)
ax.set_xticklabels(profile_summary_plot_df["shock_id"].astype(str))
ax.set_ylabel("Share")
ax.set_title(f"POSet structure summary — {FINAL_PROFILE_LEVEL}-level profile analysis")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_figure(fig, f"04_profile_structure_summary_{FINAL_PROFILE_LEVEL}levels.png", "Profile structure summary", "Comparability ratio and Pareto share.", "profile_run_summary.csv")
display(profile_summary_plot_df)

Saved table: 02_final_profile_summary_by_shock.csv
Saved figure: 04_profile_structure_summary_4levels.png


,shock_id,country_count,profile_count,pareto_country_count,comparability_ratio,incomparability_ratio,layer_count,pareto_share
1,2007,25,25,8,0.2767,0.7233,5,0.3200
4,2019,32,31,13,0.2710,0.7290,5,0.4062


In [10]:
# ------------------------------------------------------
# FIGURE 05 — PROFILE DISCRETIZATION SENSITIVITY
# ------------------------------------------------------

if not report_table_profile_sensitivity.empty:
    for shock_id, group in report_table_profile_sensitivity.groupby("shock_id"):
        group = group.sort_values("profile_levels")
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["profile_levels"], group["comparability_ratio"], marker="o", label="Comparability")
        ax.plot(group["profile_levels"], group["pareto_share"], marker="o", label="Pareto share")
        ax.axvline(FINAL_PROFILE_LEVEL, linestyle="--", linewidth=1, label=f"Selected {FINAL_PROFILE_LEVEL}")
        ax.set_title(f"Profile discretization sensitivity — shock {shock_id}")
        ax.set_xlabel("Profile levels")
        ax.set_ylabel("Share")
        ax.set_xticks(sorted(group["profile_levels"].unique()))
        ax.legend()
        ax.grid(alpha=0.25)
        save_figure(fig, f"05_profile_discretization_sensitivity_shock_{shock_id}.png", f"Profile discretization sensitivity — {shock_id}", "Comparability and Pareto share across levels.", "report_table_profile_sensitivity.csv")
else:
    print("Missing report_table_profile_sensitivity; skipping.")

Saved figure: 05_profile_discretization_sensitivity_shock_2007.png
Saved figure: 05_profile_discretization_sensitivity_shock_2019.png


In [11]:
# ------------------------------------------------------
# FIGURE 06 — EPSILON-MARGIN ROBUSTNESS
# ------------------------------------------------------

if not report_table_epsilon_margin.empty:
    for shock_id, group in report_table_epsilon_margin.groupby("shock_id"):
        group = group.sort_values("epsilon_margin")
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["epsilon_margin"], group["comparability_ratio"], marker="o", label="Comparability")
        ax.plot(group["epsilon_margin"], group["pareto_share"], marker="o", label="Frontier share")
        ax.axvline(FINAL_EPSILON_MARGIN_FOR_VISUALS, linestyle="--", linewidth=1, label=f"Reference {FINAL_EPSILON_MARGIN_FOR_VISUALS:.2f}")
        ax.set_title(f"Epsilon-margin robustness — shock {shock_id}")
        ax.set_xlabel("Epsilon margin")
        ax.set_ylabel("Share")
        ax.legend()
        ax.grid(alpha=0.25)
        save_figure(fig, f"06_epsilon_margin_robustness_shock_{shock_id}.png", f"Epsilon-margin robustness — {shock_id}", "Comparability and frontier share under margin values.", "report_table_epsilon_margin.csv")
else:
    print("Missing report_table_epsilon_margin; skipping.")

Saved figure: 06_epsilon_margin_robustness_shock_2007.png
Saved figure: 06_epsilon_margin_robustness_shock_2019.png


In [12]:
# ------------------------------------------------------
# FIGURE 07 — EPSILON TOLERANCE VS MARGIN
# ------------------------------------------------------

if not epsilon_rule_comparison.empty and "message" not in epsilon_rule_comparison.columns:
    comp = epsilon_rule_comparison[epsilon_rule_comparison["set_name"] == MAIN_SET_NAME].copy()
    for shock_id, group in comp.groupby("shock_id"):
        group = group.sort_values("epsilon_value")
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(group["epsilon_value"], group["tolerance_comparability_ratio"], marker="o", label="Tolerance")
        ax.plot(group["epsilon_value"], group["margin_comparability_ratio"], marker="o", label="Margin")
        ax.set_title(f"Epsilon rule comparison — shock {shock_id}")
        ax.set_xlabel("Epsilon value")
        ax.set_ylabel("Comparability ratio")
        ax.legend()
        ax.grid(alpha=0.25)
        save_figure(fig, f"07_epsilon_tolerance_vs_margin_shock_{shock_id}.png", f"Epsilon tolerance vs margin — {shock_id}", "Tolerance allows small disadvantages; margin does not.", "epsilon_tolerance_vs_margin_comparison.csv")
else:
    print("Missing epsilon comparison; skipping.")

Saved figure: 07_epsilon_tolerance_vs_margin_shock_2007.png
Saved figure: 07_epsilon_tolerance_vs_margin_shock_2019.png


In [13]:
# ------------------------------------------------------
# FIGURE 08 — RECOVERY VALIDATION SUMMARY
# ------------------------------------------------------

if not report_table_recovery_validation.empty:
    rv = report_table_recovery_validation[
        report_table_recovery_validation["validation_variant"].isin([PRIMARY_RECOVERY_VARIANT, STRICT_RECOVERY_VARIANT])
    ].copy()

    for shock_id, group in rv.groupby("shock_id"):
        labels = (
            group["method"].astype(str) + "\n"
            + group["setting"].astype(str) + "\n"
            + group["validation_variant"].astype(str).str.replace("_", " ")
        )
        values = group["frontier_minus_non_frontier_mean_recovery"]

        fig, ax = plt.subplots(figsize=(max(9, len(group)*1.5), 5))
        ax.bar(range(len(group)), values)
        ax.axhline(0, linewidth=1)
        ax.set_title(f"Recovery validation — shock {shock_id}")
        ax.set_ylabel("Mean recovery difference\nfrontier/Pareto minus non-frontier")
        ax.set_xticks(range(len(group)))
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.25)
        save_figure(fig, f"08_recovery_validation_summary_shock_{shock_id}.png", f"Recovery validation summary — {shock_id}", "Negative values mean frontier/Pareto countries recovered faster on average.", "report_table_recovery_validation.csv")
else:
    print("Missing report_table_recovery_validation; skipping.")

Saved figure: 08_recovery_validation_summary_shock_2007.png
Saved figure: 08_recovery_validation_summary_shock_2019.png


In [14]:
# ------------------------------------------------------
# FIGURE 09 — RECOVERY BY FINAL PROFILE LAYER
# ------------------------------------------------------

for shock_id in SHOCK_IDS:
    group = final_profile_country_map[
        (final_profile_country_map["shock_id"].astype(str) == str(shock_id))
        & (final_profile_country_map["Recovery"].notna())
    ].copy()

    if group.empty:
        print(f"No recovery/profile data for shock {shock_id}; skipping.")
        continue

    layer_stats = (
        group.groupby("poset_layer")
        .agg(recovery_median=("Recovery","median"), recovery_mean=("Recovery","mean"), country_count=("Country","nunique"))
        .reset_index()
        .sort_values("poset_layer")
    )

    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot(layer_stats["poset_layer"], layer_stats["recovery_median"], marker="o", label="Median recovery")
    ax.plot(layer_stats["poset_layer"], layer_stats["recovery_mean"], marker="o", label="Mean recovery")
    ax.set_title(f"Recovery by profile POSet layer — shock {shock_id}")
    ax.set_xlabel("POSet layer; 1 = Pareto/nondominated")
    ax.set_ylabel("Recovery years")
    ax.legend()
    ax.grid(alpha=0.25)
    save_figure(fig, f"09_recovery_by_profile_layer_shock_{shock_id}.png", f"Recovery by profile layer — {shock_id}", "Mean and median recovery by selected profile layer.", "profile_country_map.csv")
    save_table(layer_stats, f"03_recovery_by_profile_layer_shock_{shock_id}.csv", f"Recovery by profile layer — {shock_id}", "Mean and median recovery by selected profile layer.")

Saved figure: 09_recovery_by_profile_layer_shock_2007.png
Saved table: 03_recovery_by_profile_layer_shock_2007.csv
Saved figure: 09_recovery_by_profile_layer_shock_2019.png
Saved table: 03_recovery_by_profile_layer_shock_2019.csv


In [15]:
# ------------------------------------------------------
# FIGURE 10 — RECOVERY MISMATCH CASES
# ------------------------------------------------------

if not recovery_interpretation_cases.empty:
    case_counts = recovery_interpretation_cases.groupby(["shock_id","case_type"]).size().reset_index(name="case_count")
    for shock_id, group in case_counts.groupby("shock_id"):
        fig, ax = plt.subplots(figsize=(8,5))
        ax.bar(group["case_type"].astype(str), group["case_count"])
        ax.set_title(f"Structural/outcome mismatch cases — shock {shock_id}")
        ax.set_ylabel("Number of countries")
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=0.25)
        save_figure(fig, f"10_recovery_mismatch_cases_shock_{shock_id}.png", f"Recovery mismatch cases — {shock_id}", "Counts of structural/outcome mismatch cases.", "recovery_interpretation_cases.csv")
    save_table(recovery_interpretation_cases, "04_recovery_interpretation_cases.csv", "Recovery interpretation cases", "Country-level mismatch cases for discussion.")
else:
    print("Missing recovery interpretation cases; skipping.")

Saved figure: 10_recovery_mismatch_cases_shock_2007.png
Saved figure: 10_recovery_mismatch_cases_shock_2019.png
Saved table: 04_recovery_interpretation_cases.csv


In [16]:
# ------------------------------------------------------
# REPORT TABLE EXPORTS
# ------------------------------------------------------

if not sensitivity_synthesis.empty:
    save_table(sensitivity_synthesis, "05_sensitivity_synthesis.csv", "Sensitivity synthesis", "High-level synthesis from notebook 11.")
if not methodological_decisions.empty:
    save_table(methodological_decisions, "06_methodological_decision_candidates.csv", "Methodological decision candidates", "Decision-support table for final review.")
if not report_table_profile_sensitivity.empty:
    save_table(report_table_profile_sensitivity, "07_report_table_profile_sensitivity.csv", "Report table — profile sensitivity", "Profile discretization sensitivity.")
if not report_table_epsilon_margin.empty:
    save_table(report_table_epsilon_margin, "08_report_table_epsilon_margin.csv", "Report table — epsilon margin", "Epsilon-margin robustness.")
if not report_table_recovery_validation.empty:
    save_table(report_table_recovery_validation, "09_report_table_recovery_validation.csv", "Report table — recovery validation", "Recovery validation summary.")

Saved table: 05_sensitivity_synthesis.csv
Saved table: 06_methodological_decision_candidates.csv
Saved table: 07_report_table_profile_sensitivity.csv
Saved table: 08_report_table_epsilon_margin.csv
Saved table: 09_report_table_recovery_validation.csv


In [17]:
# ------------------------------------------------------
# INVENTORIES AND AUDIT WORKBOOK
# ------------------------------------------------------

figure_inventory_df = pd.DataFrame(figure_inventory)
table_inventory_df = pd.DataFrame(table_inventory)

figure_inventory_df.to_csv(PROCESSED_DIR / "final_figure_inventory.csv", index=False)
figure_inventory_df.to_csv(DIAGNOSTICS_DIR / "final_figure_inventory.csv", index=False)
table_inventory_df.to_csv(PROCESSED_DIR / "final_table_inventory.csv", index=False)
table_inventory_df.to_csv(DIAGNOSTICS_DIR / "final_table_inventory.csv", index=False)

final_visuals_run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "main_set_name": MAIN_SET_NAME,
    "final_profile_level": FINAL_PROFILE_LEVEL,
    "final_epsilon_margin_for_visuals": FINAL_EPSILON_MARGIN_FOR_VISUALS,
    "figure_count": len(figure_inventory_df),
    "table_count": len(table_inventory_df),
    "figures_dir": str(FIGURES_DIR),
    "tables_dir": str(TABLES_DIR),
    "notes": "Final visuals generated from notebooks 06–11. No scalar index created.",
}])
final_visuals_run_summary.to_csv(PROCESSED_DIR / "final_visuals_run_summary.csv", index=False)
final_visuals_run_summary.to_csv(DIAGNOSTICS_DIR / "final_visuals_run_summary.csv", index=False)

# Minimal audit workbook.
audit_xlsx = AUDIT_DIR / "12_Final_Visuals_Audit.xlsx"
with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    final_visuals_run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    figure_inventory_df.to_excel(writer, sheet_name="figure_inventory", index=False)
    table_inventory_df.to_excel(writer, sheet_name="table_inventory", index=False)
    final_profile_summary.to_excel(writer, sheet_name="profile_selection", index=False)
    if not sensitivity_synthesis.empty:
        sensitivity_synthesis.to_excel(writer, sheet_name="sensitivity", index=False)
    if not methodological_decisions.empty:
        methodological_decisions.to_excel(writer, sheet_name="decisions", index=False)

print("Audit workbook:", audit_xlsx.resolve())
display(final_visuals_run_summary)
display(figure_inventory_df)
display(table_inventory_df)

Audit workbook: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\12_Final_Visuals_Audit.xlsx


,run_id,run_timestamp,main_set_name,final_profile_level,final_epsilon_margin_for_visuals,figure_count,table_count,figures_dir,tables_dir,notes
0,20260622_142318,2026-06-22 14:23:18,baseline_6_variables,4,0.0500,17,10,D:\Milano Bicocca\Course Materials\1st Year\02...,D:\Milano Bicocca\Course Materials\1st Year\02...,Final visuals generated from notebooks 06–11. ...


,figure_file,figure_path,title,description,source_tables,created_at
0,01_project_pipeline_diagram.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Project pipeline,High-level project pipeline.,notebooks 00–12,2026-06-22 14:23:18
1,02_structural_variable_concept_map.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Structural variable concept map,Six baseline dimensions.,01_final_variable_concepts.csv,2026-06-22 14:23:18
2,03_profile_hasse_diagram_shock_2007_4levels.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Profile Hasse diagram — 2007,Transitive-reduced Hasse diagram for 4-level b...,profile_table.csv; profile_hasse_edges.csv,2026-06-22 14:23:18
3,03_profile_hasse_diagram_shock_2019_4levels.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Profile Hasse diagram — 2019,Transitive-reduced Hasse diagram for 4-level b...,profile_table.csv; profile_hasse_edges.csv,2026-06-22 14:23:18
4,04_profile_structure_summary_4levels.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Profile structure summary,Comparability ratio and Pareto share.,profile_run_summary.csv,2026-06-22 14:23:18
5,05_profile_discretization_sensitivity_shock_20...,D:\Milano Bicocca\Course Materials\1st Year\02...,Profile discretization sensitivity — 2007,Comparability and Pareto share across levels.,report_table_profile_sensitivity.csv,2026-06-22 14:23:18
6,05_profile_discretization_sensitivity_shock_20...,D:\Milano Bicocca\Course Materials\1st Year\02...,Profile discretization sensitivity — 2019,Comparability and Pareto share across levels.,report_table_profile_sensitivity.csv,2026-06-22 14:23:18
7,06_epsilon_margin_robustness_shock_2007.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Epsilon-margin robustness — 2007,Comparability and frontier share under margin ...,report_table_epsilon_margin.csv,2026-06-22 14:23:18
8,06_epsilon_margin_robustness_shock_2019.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Epsilon-margin robustness — 2019,Comparability and frontier share under margin ...,report_table_epsilon_margin.csv,2026-06-22 14:23:18
9,07_epsilon_tolerance_vs_margin_shock_2007.png,D:\Milano Bicocca\Course Materials\1st Year\02...,Epsilon tolerance vs margin — 2007,Tolerance allows small disadvantages; margin d...,epsilon_tolerance_vs_margin_comparison.csv,2026-06-22 14:23:18


,table_file,table_path,title,description,rows,columns,created_at
0,01_final_variable_concepts.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Final variable concepts,Conceptual mapping of the six baseline variables.,6,5,2026-06-22 14:23:18
1,02_final_profile_summary_by_shock.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Final profile summary by shock,Selected final profile-level metrics.,2,8,2026-06-22 14:23:18
2,03_recovery_by_profile_layer_shock_2007.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Recovery by profile layer — 2007,Mean and median recovery by selected profile l...,5,4,2026-06-22 14:23:18
3,03_recovery_by_profile_layer_shock_2019.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Recovery by profile layer — 2019,Mean and median recovery by selected profile l...,5,4,2026-06-22 14:23:18
4,04_recovery_interpretation_cases.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Recovery interpretation cases,Country-level mismatch cases for discussion.,22,9,2026-06-22 14:23:18
5,05_sensitivity_synthesis.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Sensitivity synthesis,High-level synthesis from notebook 11.,5,4,2026-06-22 14:23:18
6,06_methodological_decision_candidates.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Methodological decision candidates,Decision-support table for final review.,5,7,2026-06-22 14:23:18
7,07_report_table_profile_sensitivity.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Report table — profile sensitivity,Profile discretization sensitivity.,6,11,2026-06-22 14:23:18
8,08_report_table_epsilon_margin.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Report table — epsilon margin,Epsilon-margin robustness.,12,10,2026-06-22 14:23:18
9,09_report_table_recovery_validation.csv,D:\Milano Bicocca\Course Materials\1st Year\02...,Report table — recovery validation,Recovery validation summary.,16,14,2026-06-22 14:23:18


In [18]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("12 FINAL VISUALS COMPLETE")
print("=" * 80)

print("\nFigures folder:")
print(FIGURES_DIR.resolve())

print("\nTables folder:")
print(TABLES_DIR.resolve())

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nGenerated figures:")
for name in figure_inventory_df["figure_file"].tolist():
    print("-", name)

print("\nGenerated tables:")
for name in table_inventory_df["table_file"].tolist():
    print("-", name)

print("\nImportant notes:")
print("1. FINAL_PROFILE_LEVEL is configurable at the top.")
print("2. Current default is 4 levels for both shocks.")
print("3. Hasse diagrams show partial order structure; not scalar ranking.")
print("4. Recovery validation figures are descriptive, not causal.")

print("\nNext stage:")
print("Report writing / final narrative.")

12 FINAL VISUALS COMPLETE

Figures folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Final_Visuals

Tables folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Tables\Final_Visuals

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Final_Visuals

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\12_Final_Visuals

Generated figures:
- 01_project_pipeline_diagram.png
- 02_structural_variable_concept_map.png
- 03_profile_hasse_diagram_shock_2007_4levels.png
- 03_profile_hasse_diagram_shock_2019_4levels.png
- 04_profile_structure_summary_4levels.png
- 05_profile_discretization_sensitivity_shock_2007.png
- 05_profile_discretization_sensit